#### Igor Oliveira Souza e Silva - 12311BCC017  
#### João Pedro Costa Baroni - 12311BCC035
---

1. Um call center recebe ligações seguindo um processo de Poisson com taxa de 12 ligações por hora. O tempo que o atendente leva para resolver um problema é exponencialmente distribuído, com média de 4 minutos. Modele o número de ligações na fila (incluindo a em atendimento).

(b) Qual é o número médio de clientes no sistema (fila + atendimento) em estado estacionário?

(c) Qual é o tempo médio que um cliente gasta no sistema?

In [44]:
import numpy as np

lmbd = 12 / 60
mu = 1 / 4   
n = 100000

tempos_entre_chegadas = np.random.exponential(1 / lmbd, n)

tempos_de_servico = np.random.exponential(1 / mu, n)

chegadas = np.cumsum(tempos_entre_chegadas)

inicios_atendimento = np.zeros(n)
fins_atendimento = np.zeros(n)

inicios_atendimento[0] = chegadas[0]
fins_atendimento[0] = inicios_atendimento[0] + tempos_de_servico[0]

for i in range(1, n):

    inicios_atendimento[i] = max(chegadas[i], fins_atendimento[i-1])

    fins_atendimento[i] = inicios_atendimento[i] + tempos_de_servico[i]

tempo = fins_atendimento - chegadas


W = np.mean(tempo)

lambda_efetivo = n / fins_atendimento[-1] 
L = lambda_efetivo * W

print(f"(b) Número médio de clientes no sistema (L): {L:.4f} clientes")
print(f"(c) Tempo médio que um cliente gasta no sistema (W): {W:.4f} minutos")

(b) Número médio de clientes no sistema (L): 3.8550 clientes
(c) Tempo médio que um cliente gasta no sistema (W): 19.2863 minutos


6. Compare o tempo de espera (W) de duas filas M/M/1 independentes com taxa de chegada λ = 1 cada uma com o de uma fila única M/M/2 com taxa de chegada λ = 2. Calcule o tempo médio que um atendente fica ocioso em cada caso. Assuma a mesma taxa de serviço para todos os atendentes.

In [ ]:
def mm1(lmdb, mu, n):
    chegadas = np.random.exponential(1 / lmdb, n)
    servicos = np.random.exponential(1 / mu, n)
    momentos_chegada = np.cumsum(chegadas)
    
    inicios = np.zeros(n)
    fins = np.zeros(n)
    inicios[0] = momentos_chegada[0]
    fins[0] = inicios[0] + servicos[0]
    
    for i in range(1, n):
        inicios[i] = max(momentos_chegada[i], fins[i-1])
        fins[i] = inicios[i] + servicos[i]
        
    tempo_no_sistema = fins - momentos_chegada
    w_medio = np.mean(tempo_no_sistema)
    
    tempo_total = fins[-1]
    tempo_trabalhado = np.sum(servicos)
    ociosidade = 1 - (tempo_trabalhado / tempo_total)
    
    return w_medio, ociosidade*100

def mm2(lmdb, mu, n):
    chegadas = np.random.exponential(1 / lmdb, n)
    servicos = np.random.exponential(1 / mu, n)
    momentos_chegada = np.cumsum(chegadas)

    fins = np.zeros(n)
    livre_s1 = 0.0
    livre_s2 = 0.0
    
    for i in range(n):
        chegada = momentos_chegada[i]
        servico = servicos[i]
        
        if livre_s1 <= livre_s2:
            inicio = max(chegada, livre_s1)
            fim = inicio + servico
            livre_s1 = fim
        else:
            inicio = max(chegada, livre_s2)
            fim = inicio + servico
            livre_s2 = fim
            
        fins[i] = fim
        
    tempo_no_sistema = fins - momentos_chegada
    w_medio = np.mean(tempo_no_sistema)
    
    tempo_total = max(livre_s1, livre_s2)
    tempo_trabalhado = np.sum(servicos)
    ociosidade = 1 - (tempo_trabalhado / (2 * tempo_total)) 
    
    return w_medio, ociosidade*100


clientes = 100000
mu = 1.5

w_mm1, ociosidade_mm1 = mm1(1.0, mu, clientes)
w_mm2, ociosidade_mm2 = mm2(2.0, mu, clientes)

print(f"M/M/1: W = {w_mm1:.4f} | Ociosidade do atendente = {ociosidade_mm1:.2f}%")
print(f"M/M/2: W = {w_mm2:.4f} | Ociosidade do atendente = {ociosidade_mm2:.2f}%")

M/M/1: W = 1.9871 | Ociosidade do atendente = 33.29%
M/M/2: W = 1.2177 | Ociosidade do atendente = 33.31%
